<img src="https://cads.ai/image/CADSLogo.png" style="height: 100px;" align=left>
<div align=right><svg height=100px viewBox="0 0 24 24" fill="none" xmlns="http://www.w3.org/2000/svg"><g clip-path="url(#clip1_4783_495)"><path d="M6.64803 14.103C7.89438 12.8566 8.595 11.1643 8.595 9.40177C8.595 7.63928 7.89378 5.94697 6.64803 4.70058L1.94697 0C0.701225 1.24639 0 2.9387 0 4.70119C0 6.46368 0.701225 8.15599 1.94697 9.40238L6.64742 14.103H6.64803Z" fill="#7B9AB5"></path><path d="M16.4845 14.5379C15.2388 13.2921 13.5459 12.5908 11.7841 12.5908C10.0222 12.5908 8.32936 13.2921 7.08301 14.5379L11.7841 19.239C13.0298 20.4848 14.7227 21.1861 16.4851 21.1861C18.2476 21.1861 19.9398 20.4848 21.1862 19.239L16.4851 14.5379H16.4845Z" fill="#7B9AB5"></path><path d="M1.95832 19.228C3.20468 20.4738 4.89693 21.1751 6.65938 21.1751V14.5269H0.0107422C0.0113472 16.2893 0.711968 17.9817 1.95832 19.228Z" fill="#7B9AB5"></path><path d="M18.2997 7.58717C17.0533 6.34138 15.3611 5.63953 13.598 5.64014C11.8356 5.64014 10.1433 6.34138 8.89697 7.58777L13.598 12.289L18.2997 7.58717Z" fill="#7B9AB5"></path></g></svg></div>

<a box-sizing="border-box" color-scheme=dark font-family="IBM Plex Sans" line-height=30px font-size=72px font-weight=700 align=center>© 2026 — Mehdi Lotfinejad ⨯ Azhar bin Ab Halim @ DATATWEETS</a>

# Meet the helpdesk: a loop that always finishes
Day 1 | Notebook 1 of 4

![A car drives round a ring road past a striped check gate and a fuel gauge that stands for the stop budget, while a flag in the middle asks if the task is done.](https://github.com/saifulak/EAE/blob/main/LLM/media/d1-01-hero-loop-gate.svg?raw=1)

**You can re-run any practice until it works. Nothing here is recorded or marked.**

**Today in one sentence:** you build a small AI helper that always finishes, because it checks its own answer and stops when its budget runs out.

## Today's story

Seri Maju Berhad is a **fictional** company. It has about 1,200 staff, a head office in Kuala Lumpur and plants in Penang and Johor Bahru. You have joined its HR and finance helpdesk team. Staff send questions about leave, claims and travel.

*Over Day 1 you build a small AI helper that answers them safely. All money is in ringgit (RM).

> ! **WARNING**
> The people, policies and amounts are invented.
> *Fictional example, not legal or tax advice.*

## What you will be able to do

1. Write a definition of done, one automatic check and a stop budget for a small AI task, then run it.
2. Call a chat model, read the answer, and estimate tokens and cost (*eg.*, in RM).

---

**Prerequisites:** basic Python (functions, lists, dicts).

**Words you do not know?** Open the [Glossary](glossary.md).


In [1]:
# Answer switch (change it, then choose Run > Restart Kernel and Run All Cells).
#     False = you type the blanks.
#     True = Run All reads the reference answers.
USE_SOLUTIONS = globals().get("USE_SOLUTIONS", False)

In [2]:
%pip install "langchain-groq>=0.3,<0.4"

In [5]:
# ---- CANONICAL SETUP ----
# ---- HELPER - Run it, then ignore it. ----
from datetime import datetime

from IPython.display import Markdown


## You can re-run any practice until it works. Nothing here is recorded or marked.
# Standard-library helpers for settings files, timing and folder paths.
import json
import os
import sys
import time
from glob import glob
from pathlib import Path
from typing import Literal

# Third-party pieces: HTTP client, Groq chat model, and LangChain base classes for models and embeddings.
import httpx
import sklearn
from dotenv import load_dotenv
from langchain_core.caches import InMemoryCache
from langchain_core.embeddings import Embeddings
from langchain_core.language_models.chat_models import BaseChatModel
from langchain_core.language_models.fake_chat_models import FakeListChatModel
from langchain_core.messages import AIMessage
from langchain_core.outputs import ChatGeneration, ChatResult
from langchain_groq import ChatGroq

# Patch to prevent: AttributeError: module 'langchain' has no attribute 'verbose', 'debug', or 'llm_cache'
import langchain
langchain.verbose = False
langchain.debug = False
langchain.llm_cache = None

# scikit-learn powers the offline TF-IDF embeddings; stop early with an install hint if it is missing.
try:
    from sklearn.feature_extraction.text import TfidfVectorizer
except ImportError as e:
    raise ImportError(
        "scikit-learn is needed for TfidfEmbeddings. In a terminal run: "
        "pip install scikit-learn  (then restart the kernel)."
    ) from e

# Load a default .env file, if there is one, into the environment.
load_dotenv()


SOLUTIONS = {}  # step_id -> (builder function, one-sentence why, code text shown by show_solution)
CHECKPOINTS = {}  # part_id -> function(ns) that rebuilds the variables a late learner needs (canned/mock only)
CODES = {}  # part -> reference code texts, in order

# Look up per-million-token prices for the chosen GROQ_MODEL, used for cost estimates.
# MYR is preferred, then USD, then a built-in default.
with open("_groq_model_per_million_prices.json", mode="rb") as _model_prices_json:
    _MODEL_PRICES = json.load(_model_prices_json).get(
        os.environ.get("GROQ_MODEL", ""), {}
    )
    model_input_price = (_model_input_price := _MODEL_PRICES.get("input", {})).get(
        "MYR", _model_input_price.get("USD", 0.50)
    )
    model_output_price = (_model_output_price := _MODEL_PRICES.get("output", {})).get(
        "MYR", _model_output_price.get("USD", 2.00)
    )


def register(step_id, why, code=""):
    """Decorator used by the notebook: SOLUTIONS['d1-01-p2'] = the reference builder."""

    def deco(fn):
        SOLUTIONS[step_id] = (fn, why, code or getattr(fn, "__doc__", "") or "")
        return fn

    return deco


def checkpoint(part_id):
    """Decorator: register the function that rebuilds this Part's variables (see fast_forward)."""

    def deco(fn):
        CHECKPOINTS[part_id] = fn
        return fn

    return deco


def pick(step_id, attempt=None, is_using_solutions: bool = True):
    """Use your attempt, or the reference answer if USE_SOLUTIONS is True or your attempt is None."""
    if is_using_solutions or attempt is None:
        if step_id not in SOLUTIONS:
            raise KeyError(
                f"No reference answer registered for '{step_id}'. Click the heading 'Part A', "
                "choose Run > Run All Above Selected Cell, then run this cell again."
            )
        return SOLUTIONS[step_id][0]()
    return attempt


def check(step_id, ok, hint=""):
    """Plain self-check: tick or cross plus a hint. Unlimited re-runs. Nothing is stored or counted."""
    print(
        ("\u2714 " + step_id + ": looks right")
        if ok
        else (
            "\u2718 "
            + step_id
            + ": not yet. "
            + (hint or "Re-read the step, change your code, run again.")
        )
    )
    return bool(ok)


def show_solution(step_id):
    """Print the reference code and why it works. Keyboard-friendly: no copying needed, use pick()."""
    if step_id not in SOLUTIONS:
        print("No reference answer registered for", step_id)
        return
    _, why, code = SOLUTIONS[step_id]
    print(code.rstrip() or "(see the reference cell)")
    print("Why:", why)


def fast_forward(part_id):
    """Late or stuck? Rebuild this Part's variables from reference code (canned/mock, no API calls)."""

    if part_id not in CHECKPOINTS:
        print(
            "Not ready yet. Click the heading 'Part A', choose Run > Run All Above Selected Cell, "
            "then run this cell again. Known parts:",
            ", ".join(sorted(CHECKPOINTS)) or "(none registered yet)",
        )
        return

    if globals().get("LLM_MODE") == "groq":
        print("Note: fast-forward uses canned replies, not the live model.")

    CHECKPOINTS[part_id](globals())
    print("\u2714 Caught up to", part_id, "- continue with the next cell.")


# Load the workplace policy texts as system messages so every answer can be grounded in them.
def get_system_corpus(context: Literal["workplace"]) -> list[tuple[str, str]]:
    _corpus_dir: Path
    system_prompt: list[tuple[str, str]] = []

    if context == "workplace":
        # Support both the original courseware layout and Google Colab uploads.
        _patterns = [
            str(Path.cwd() / "data" / "workplace*" / "*.txt"),
            str(Path.cwd() / "workplace_corpus" / "sm-pol-*.txt"),
            str(Path.cwd() / "sm-pol-*.txt"),
            "/content/workplace_corpus/sm-pol-*.txt",
            "/content/sm-pol-*.txt",
        ]
    else:
        raise ValueError(f"Unsupported corpus: '{context}'")

    # Collect matching policy files, avoiding duplicates.
    _policy_paths = sorted({Path(p) for pattern in _patterns for p in glob(pattern)})
    for _corpus_pathname in _policy_paths:
        with open(_corpus_pathname, "r", encoding="utf-8") as _corpus_file:
            policy_id = _corpus_pathname.stem.replace("sm-pol-", "").upper()
            system_prompt.append(("system", f"Policy ID: {policy_id}\n{_corpus_file.read()}"))

    # Finish with a rule that makes answers cite their source identifier.
    system_prompt.append(
        (
            "system",
            "Assistant should always cite source identifier to support claim when answering any asks.",
        )
    )

    return system_prompt


# Remember a code snippet under its Part so a late learner can rebuild that Part.
def state(part, code):
    CODES.setdefault(part, []).append(code.strip())


# Register a reference answer: keep its code text and a builder that runs it and returns result.
def ref(part, step, why, code):
    code = code.strip()
    if part:
        CODES.setdefault(part, []).append(code)

    # Run the reference code in a copy of the notebook variables and hand back its result.
    def build():
        ns = dict(globals())
        exec(code, ns)
        return ns["result"]

    register(step, why, code)(build)


# Define how to catch up: re-run the stored code of the listed Parts in order (used by fast_forward).
def rebuild(part, *parts):
    checkpoint(part)(lambda ns: [exec(c, ns) for p in parts for c in CODES[p]])


# Build the shared system messages once so later cells can reuse them.
SYSTEM_PROMPT = get_system_corpus("workplace")


# Guard the Python version: 3.14 or newer is too new for some course packages.
if sys.version_info[:2] >= (3, 14):
    raise RuntimeError(
        f"Python {sys.version.split()[0]} is too new for this course: some packages are not ready. "
        "Create the environment with Python 3.12 (3.11-3.13 work) and pick it as the notebook kernel."
    )

# Also stop on versions older than the supported 3.11.
assert (3, 11) <= sys.version_info[:2] <= (3, 13), (
    f"Use Python 3.11-3.13 (you have {sys.version.split()[0]})"
)
# Key lookup order:
#   1. file named in LM_ENV_FILE
#   2. ~/.langchain-mastery/.env (local, not synced)
#   3. .env next to notebooks.
# Only these four names are read from the file; everything else in it is ignored.

# Names read from the settings file, and folder names that suggest cloud-synced storage.
_ALLOWED = ("GROQ_API_KEY", "GROQ_MODEL", "LLM_MODE", "LM_ENV_FILE")
_SYNCED = ("onedrive", "sharepoint", "dropbox", "google drive", "googledrive", "icloud")


def _tilde(
    p,
):  # show the home folder as ~ so a shared screen does not show your user name
    p, home = str(p), str(Path.home())
    return (
        "~" + p[len(home) :]
        if p.lower().startswith(home.lower())
        else "(outside home) " + Path(p).name
    )


# Warn early if LM_ENV_FILE points to a file that does not exist.
_lm = os.getenv("LM_ENV_FILE", "").strip()
if _lm and not Path(_lm).is_file():
    print(
        "NOTE: LM_ENV_FILE is set but that file was not found, so it is ignored. Check the path."
    )

# Candidate settings files in lookup order; the first one that exists is used.
_env_files = [
    _lm,
    str(Path.home() / ".langchain-mastery" / ".env"),
    str(Path.cwd() / ".env"),
    str(Path.cwd() / "courseware" / ".env"),
]

# Copy only the allowed names from the chosen file into the environment, never overwriting values already set.
_env_file = next((p for p in _env_files if p and Path(p).is_file()), None)
if _env_file:
    try:
        from dotenv import dotenv_values

        for _k, _v in dotenv_values(_env_file).items():
            if (
                _k in _ALLOWED and _v is not None and _k not in os.environ
            ):  # existing variables win
                os.environ[_k] = _v

        # Tell the learner which file was used, with the home folder hidden.
        print(
            "Settings file:",
            _tilde(_env_file),
            "(only",
            ", ".join(_ALLOWED),
            "are read)",
        )

        # Warn if the file sits in a synced folder, where a key could be shared by accident.
        if any(w in _env_file.lower() for w in _SYNCED):
            print(
                "WARNING: your .env is in a synced folder (OneDrive, SharePoint, Dropbox, Google Drive "
                "or iCloud). Move it to ~/.langchain-mastery/.env"
            )

    except ImportError:
        print("python-dotenv not installed: using system environment variables only")
else:
    # With no settings file the notebook still runs in mock mode, so no key is needed.
    print("No .env file found: mock mode, no key needed.")

# Choose mock (offline, canned replies) or groq (live model); mock is the default.
LLM_MODE = os.getenv("LLM_MODE", "mock").strip().lower()
if LLM_MODE not in ("mock", "groq"):
    raise ValueError("LLM_MODE must be 'mock' or 'groq' (check your .env file)")

# Tracing switch: mock mode forces every tracing variable off; groq mode only warns if one is on.
_TRACING = ("LANGCHAIN_TRACING_V2", "LANGCHAIN_TRACING", "LANGSMITH_TRACING")
if LLM_MODE == "mock":
    for _t in _TRACING:
        os.environ[_t] = "false"

    print(
        "Mock mode: LangChain tracing is switched off for this session (nothing is sent anywhere)."
    )

elif any(os.getenv(_t, "").strip().lower() == "true" for _t in _TRACING):
    print(
        "WARNING: a LangChain tracing variable is on: live runs may be sent to a tracing service. Set it to false."
    )


class ScriptedToolChat(BaseChatModel):
    """Mock model for tool calling: replays a script. A str is a final answer; a dict
    {'name':..., 'args':{...}} (or a list of them) becomes AIMessage.tool_calls. Canned, not real."""

    # The script to replay and the position of the next step.
    script: list = []
    pos: int = 0

    @property
    def _llm_type(self):
        return "scripted-tool-chat"

    def bind_tools(self, tools, **kwargs):
        return self  # tools are ignored: the script decides

    def _generate(self, messages, stop=None, run_manager=None, **kwargs):
        # Take the next scripted step, looping back to the start when the script runs out.
        step = (
            self.script[self.pos % len(self.script)] if self.script else "(mock reply)"
        )
        self.pos += 1

        # A text step becomes a final answer; a dict step becomes tool calls with made-up ids.
        if isinstance(step, str):
            msg = AIMessage(content=step)

        else:
            calls = step if isinstance(step, list) else [step]
            msg = AIMessage(
                content="",
                tool_calls=[
                    {
                        "name": c["name"],
                        "args": c.get("args", {}),
                        "id": f"mock_{self.pos}_{i}",
                        "type": "tool_call",
                    }
                    for i, c in enumerate(calls)
                ],
            )

        return ChatResult(generations=[ChatGeneration(message=msg)])


def get_llm(responses=None, tool_calls=None):
    """Return a chat model. Groq in 'groq' mode; offline canned mock otherwise."""

    if LLM_MODE == "groq":
        # Live mode needs both a model name and a key; otherwise explain what is missing.
        model = os.getenv("GROQ_MODEL", "").strip()

        if not model or not os.getenv("GROQ_API_KEY", "").strip():
            raise RuntimeError(
                "Groq mode needs GROQ_API_KEY and GROQ_MODEL in .env "
                "(suggested model: openai/gpt-oss-20b; confirm in the Groq console), "
                "or set LLM_MODE=mock."
            )

        # Return the Groq model; temperature 0 and a cache keep replies repeatable.
        print("LIVE model:", model)
        return ChatGroq(
            cache=InMemoryCache(),
            model=model,
            temperature=0,
            http_client=httpx.Client(verify=False),
        )

    # Mock mode: an offline fake model, with scripted tool calls if given, otherwise canned text replies.
    print("MOCK: canned replies")
    if tool_calls:
        return ScriptedToolChat(script=list(tool_calls))

    return FakeListChatModel(responses=list(responses or ["(mock reply)"]))


class TraceLog:
    """Tiny local trace: one dict per step. This class sends nothing anywhere."""

    RATE_PER_1K_TOKENS = 0.0005  # PLACEHOLDER rate in RM per 1,000 tokens (not a real price); use your plan's

    def __init__(self):
        self.rows = []

    def add(self, step, tool="-", tokens_in=0, tokens_out=0, seconds=0.0):
        # Estimate cost from the total tokens at the placeholder rate (not a real price).
        cost = (tokens_in + tokens_out) / 1000 * self.RATE_PER_1K_TOKENS

        # Store one row per step so the trace can be shown as a table.
        self.rows.append(
            {
                "step": step,
                "tool": tool,
                "tokens_in": tokens_in,
                "tokens_out": tokens_out,
                "seconds": round(seconds, 3),
                "est_cost_myr": round(cost, 6),
            }
        )

    def timed(self, step, fn, *args, tool="-", **kw):
        # Run fn, time it, and log a step; token counts are rough guesses (about 4 characters per token).
        t0 = time.perf_counter()
        out = fn(*args, **kw)

        self.add(
            step,
            tool,
            len(str(args)) // 4,
            len(str(out)) // 4,
            time.perf_counter() - t0,
        )

        return out

    # Sum tokens, seconds and estimated cost over all recorded steps.
    def total(self):
        return {
            k: round(sum(r[k] for r in self.rows), 6)
            for k in ("tokens_in", "tokens_out", "seconds", "est_cost_myr")
        }


class TfidfEmbeddings(Embeddings):
    """Offline embeddings: scikit-learn TF-IDF fitted on your corpus, L2-normalised."""

    def __init__(self, corpus):
        # Fit the vocabulary on the corpus once so documents and questions share one vector space.
        self.vec = TfidfVectorizer(norm="l2", lowercase=True).fit(list(corpus))

    def embed_documents(self, texts):
        return self.vec.transform(list(texts)).toarray().tolist()

    def embed_query(self, text):
        return self.embed_documents([text])[0]


# ---- END HELPER ----
# ---- END CANONICAL SETUP ----

No .env file found: mock mode, no key needed.
Mock mode: LangChain tracing is switched off for this session (nothing is sent anywhere).


## Three rules before you start

1. **Keep your key private.** Never paste a key into a notebook, a chat or an email. It lives only in your `.env` file.
2. **Do not paste personal or company data.** In live mode your text goes to a third party (Groq). Use only this course's fictional data.
3. **AI output needs a human check.** A fluent answer can still be wrong. You check it before anyone relies on it.

Mock mode is the default. It needs no key and sends nothing anywhere.

If either code cell above stopped with a red message, read the message, then open the PRE-WORK section below.

<details>
<summary><strong>PRE-WORK:</strong> install, key file, kernel, proxy</summary>

**Quick start without Jupyter:** the first lines of `courseware/.requirements.txt` and `courseware/.env.tmp` say the same in about ten lines. Open either file in Notepad and follow it before class.

1. Install **Python 3.11, 3.12 or 3.13**. Python 3.14 is too new for these packages.
2. Make a **virtual environment** (a private folder of packages for this course). In a terminal, in the folder that holds this notebook:

   ```powershell
   python -m venv .venv
   .\.venv\Scripts\activate
   pip install -r .\.requirements.txt
   ```

   or

   ```shell
   python -m venv .venv
   source ./.venv/bin/activate
   pip install -r ./.requirements.txt
   ```

3. **Key file (only for live mode).** Rename `.env.tmp` to `.env`. Names that start with a dot may be hidden: turn on Hidden items in File Explorer.
4. **Mock or live.** `LLM_MODE=mock` (default) runs offline with canned replies. `LLM_MODE=groq` calls Groq. Suggested `GROQ_MODEL`: `openai/gpt-oss-20b`. Free-plan limits applies; check the Groq console.
5. **Kernel** (the program that runs your notebook code). Pick the `.venv` Python as the kernel. If it is missing from the list, run `python -m ipykernel install --user --name llm-mastery`.
6. **Proxy** (a company gateway between your PC and the internet). If `pip` or live calls time out, ask IT for your proxy address and set `HTTPS_PROXY`. Mock mode needs no internet after the install.
7. **Tracing.** Do not set LangSmith or other tracing variables for this course. Mock mode switches LangChain tracing off for you.
8. **Open item:** Tell your instructor if something breaks.
</details>

## How the practices work

- A practice cell has a **starter** you can edit and a line like `my_attempt = None`. **If you leave it as `None`, the notebook quietly runs the reference answer instead, so Run All works even when you wrote nothing.** To learn, write your own and set `my_attempt` to it: then yours runs.
- `check(step_id, ...)` prints ✔ (`looks right`) or ✘ (`not yet`) and a hint. Run it as often as you like.
- Stuck? `show_solution(step_id)` prints the reference code.
- Honest note: reading the answer first and then trying your own; this teaches more than skipping it. The collapsed cells just under the set-up cells only register reference answers: run them with everything above.

In [6]:
# Notebook-specific data path and a quick setup check (mock mode needs no key)

DATA = Path("data")

# Show the Python version and mode first, so setup problems are easy to spot.
print("Python", sys.version.split()[0], "| LLM_MODE =", LLM_MODE)
print("Data folder:", DATA if DATA else "NOT FOUND: open from courseware/")

# Confirm scikit-learn is installed; the course helpers need it.
try:
    print("scikit-learn", sklearn.__version__)

except ImportError:
    print("scikit-learn missing: run pip install scikit-learn, then restart the kernel")

# Stop early with a clear message if the data folder is missing.
assert DATA is not None, "Data folder not found"

Python 3.13.16 | LLM_MODE = mock
Data folder: data
scikit-learn 1.6.1


In [7]:
# Helper (collapsed): ref() stores reference code for show_solution and pick.
# Tell fast_forward how to rebuild each Part's variables for a late learner (Part B also needs Part A).
rebuild("d1-01-A", "d1-01-A")
rebuild("d1-01-B", "d1-01-A", "d1-01-B")

# Keep Part A's data-loading code so it can be replayed during fast_forward.
state(
    "d1-01-A",
    """
POL = {p.name[:9].upper(): p.read_text(encoding='utf-8')
       for p in sorted(DATA.glob('sm-pol-*.txt') if list(DATA.glob('sm-pol-*.txt')) else (DATA / 'workplace_corpus').glob('sm-pol-*.txt'))}
Q = 'How many unused leave days can I carry forward?'
""",
)

# Keep Part B's price table code so it can be replayed during fast_forward.
state(
    "d1-01-B",
    "PRICES_MYR_PER_1M = {"
    + f"'input': {model_input_price}, 'output': {model_output_price}"
    + "}   # PLACEHOLDER, not real",
)

# Reference answer for Step 1: where the answer text sits in the reply.
ref(
    None,
    "d1-01-call",
    "The answer text lives in reply.content.",
    "llm = get_llm(responses=['Up to 5 unused days carry forward (SM-POL-01).'])\n"
    "result = llm.invoke(SYSTEM_PROMPT + [('human', Q)]).content",
)

# Reference answer for Step 2: the automatic check that decides whether the reply is good enough.
ref(
    "d1-01-A",
    "d1-01-check",
    "Lower-case, then look for the id and the 5.",
    "def is_done(text):\n"
    "    t = text.lower()\n"
    "    return 'sm-pol-01' in t and '5' in t\n\n"
    "result = is_done",
)

# Reference answer for Practice 1: a finished three-part contract.
ref(
    None,
    "d1-01-contract",
    "All three are written before the run.",
    "result = {\n"
    "    'done': 'Names SM-POL-01 and says up to 5 days carry forward.',\n"
    "    'check': 'Look for the id and the number 5.',\n"
    "    'max_calls': 3\n"
    "}",
)

# Reference answer for Practice 2: the loop that stops on success or when the budget runs out.
ref(
    "d1-01-A",
    "d1-01-loop",
    "Return when the check is true; else stop at budget.",
    "def run_loop(llm, question, is_done, max_calls=3):\n"
    "    replies = []\n"
    "    for call in range(1, max_calls + 1):\n"
    "        text = llm.invoke(question).content\n"
    "        replies.append(text)\n"
    "        if is_done(text):\n"
    "            return {'done': True, 'calls': call, 'replies': replies}\n"
    "    return {'done': False, 'calls': max_calls, 'replies': replies}\n\n"
    "result = run_loop",
)

# Reference answer for Part B Step 1: the token estimator.
ref(
    "d1-01-B",
    "d1-01-estimate",
    "About 4 characters per token; at least 1.",
    "def estimate_tokens(text):\n"
    "    return max(1, len(text) // 4)\n\n"
    "result = estimate_tokens",
)

# Reference answer for Part B Step 2: use the exact usage record when there is one, else estimate.
ref(
    "d1-01-B",
    "d1-01-usage",
    "Use the exact record, else estimate.",
    "def usage_of(msg, prompt):\n"
    "    um = msg.usage_metadata\n"
    "    if um:\n"
    "        return um['input_tokens'], um['output_tokens'], 'usage_metadata'\n\n"
    "    return estimate_tokens(prompt), estimate_tokens(msg.content), 'estimate'\n\n"
    "result = usage_of",
)

# Reference answer for the Part B practice: turn token counts into a cost in RM.
ref(
    "d1-01-B",
    "d1-01-cost",
    "Each side has its own price per million.",
    "def cost_myr(tokens_in, tokens_out, prices=PRICES_MYR_PER_1M):\n"
    "    return (tokens_in * prices['input'] + tokens_out * prices['output']) / 1_000_000\n\n"
    "result = cost_myr",
)

## The Finishable Loop

### Idea: A loop needs three things written down before it runs (●)

An AI **loop** asks the model, checks the answer, and asks again if the answer is not good enough. A loop with no end wastes time and money. So write these down first:

| Part | Plain meaning | Helpdesk example |
|---|---|---|
| Definition of done | One sentence: what a good answer looks like | The reply names policy `SM-POL-01` and says 5 days |
| Automatic check | A small function that returns True or False | `'sm-pol-01' in text.lower()` |
| Stop budget | A hard limit so the loop cannot run on | At most 3 model calls |

### Idea: One call through `get_llm()` (●)

A **chat model** takes text in and gives a message back. `llm.invoke(text)` returns that message. The answer text is in `reply.content`. `get_llm()` hides which provider you use.

**In **mock** mode it replays canned replies you give it: they prove your code works, not that a model is smart.*

**Step 1 - first call. (●)** Staff ask the helpdesk about leave. Read the reply text out of the message. In mock mode the reply is canned.

In [8]:
%pip install "langchain>=0.3,<0.4" "langchain-core>=0.3,<0.4" "langchain-groq>=0.3,<0.4" "langchain-text-splitters>=0.3,<0.4" "langgraph>=0.6,<0.7" "pydantic>=2,<3" "scikit-learn>=1.4" "python-dotenv"

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 29.6 MB/s  0:00:00
  Attempting uninstall: langgraph-sdk
    Found existing installation: langgraph-sdk 0.4.8
    Uninstalling langgraph-sdk-0.4.8:
      Successfully uninstalled langgraph-sdk-0.4.8
  Attempting uninstall: langgraph-checkpoint
    Found existing installation: langgraph-checkpoint 4.3.0
    Uninstalling langgraph-checkpoint-4.3.0:
      Successfully uninstalled langgraph-checkpoint-4.3.0
  Attempting uninstall: langgraph-prebuilt
    Found existing installation: langgraph-prebuilt 1.1.1
    Uninstalling langgraph-prebuilt-1.1.1:
      Successfully uninstalled langgraph-prebuilt-1.1.1
  Attempting uninstall: langchain
    Found existing installation: langchain 1.4.6
    Uninstalling langchain-1.4.6:
      Successfully uninstalled langchain-1.4.6
  Attempting uninstall: langgraph
    Found existing installation: langgraph 1.2.16
    Uninstalling langgraph-1.2.16:
      Successfully uninstalled langgraph-1.2.16
   ━━━━

In [9]:
import sys
import langchain
import langchain_core
import langchain_groq

print("Python:", sys.version)
print("LangChain:", getattr(langchain, "__version__", "unknown"))
print("LangChain Core:", getattr(langchain_core, "__version__", "unknown"))
print("LangChain Groq:", getattr(langchain_groq, "__version__", "unknown"))
print("LangChain file:", langchain.__file__)
print("Has verbose:", hasattr(langchain, "verbose"))

Python: 3.13.16 (main, Oct  1 2026, 15:40:24) [GCC 13.3.0]
LangChain: 1.4.6
LangChain Core: 0.3.86
LangChain Groq: 0.3.8
LangChain file: /usr/local/lib/python3.13/dist-packages/langchain/__init__.py
Has verbose: True


In [10]:
print("DATA:", "DATA" in globals())
print("get_llm:", "get_llm" in globals())
print("pick:", "pick" in globals())
print("check:", "check" in globals())
print("USE_SOLUTIONS:", "USE_SOLUTIONS" in globals())

DATA: True
get_llm: True
pick: True
check: True
USE_SOLUTIONS: True


In [11]:
# Load the policy files into a lookup by policy id, so the helpdesk has real text to answer from.
POL = {
    p.name[:9].upper(): p.read_text(encoding="utf-8")
    for p in sorted(DATA.glob("sm-pol-*.txt") if list(DATA.glob("sm-pol-*.txt")) else (DATA / "workplace_corpus").glob("sm-pol-*.txt"))
}

# The staff question used throughout this part.
Q = "How many unused leave days can I carry forward?"

# Use a mock model with one canned reply, so the demo repeats the same way and needs no key.
llm = get_llm(responses=["Up to 5 unused days carry forward (SM-POL-01)."])

# Send the policies plus the question to the model; the result is a message object, not plain text.
reply = llm.invoke(SYSTEM_PROMPT + [("human", Q)])

# Your turn: pull the answer text out of the reply message.
my_answer = reply.content

# Use your answer, or the reference answer if you left the blank empty.
answer = pick("d1-01-call", my_answer, is_using_solutions=USE_SOLUTIONS)

# Self-check: the answer must be non-empty text.
check(
    "d1-01-call",
    isinstance(answer, str) and len(answer) > 0,
    "The text is in reply.content (a string).",
)

# Show the answer as a quoted block so it is easy to read.
print("answer:")
Markdown("> " + answer.replace("\n", "\n> "))

MOCK: canned replies
✔ d1-01-call: looks right
answer:


> Up to 5 unused days carry forward (SM-POL-01).

In [12]:
from pathlib import Path
from IPython.display import Markdown, display
from langchain_core.language_models.fake_chat_models import FakeListChatModel

# Load policy files
DATA = Path("/content")

POL = {
    p.stem.replace("sm-pol-", "").upper(): p.read_text(encoding="utf-8")
    for p in sorted(DATA.glob("sm-pol-*.txt"))
}

print("Policies loaded:", len(POL))

# Create the system prompt from policy text
SYSTEM_PROMPT = [
    ("system", f"Policy ID: {policy_id}\n{text}")
    for policy_id, text in POL.items()
]

SYSTEM_PROMPT.append((
    "system",
    "Answer using the supplied policies and cite the relevant policy ID."
))

# Ask the question
Q = "How many unused leave days can I carry forward?"

# Mock model: fixed reply, no API key required
llm = FakeListChatModel(
    responses=["Up to 5 unused days carry forward (SM-POL-01)."]
)

reply = llm.invoke(SYSTEM_PROMPT + [("human", Q)])

# Extract and display the response
answer = reply.content

assert isinstance(answer, str) and answer.strip(), "Empty response!"

print("Answer:")
display(Markdown("> " + answer))

assert "SM-POL-01" in answer
assert "5" in answer
print("All checks passed!")

Policies loaded: 16
Answer:


> Up to 5 unused days carry forward (SM-POL-01).

All checks passed!


In [13]:
from pathlib import Path
from IPython.display import Markdown

DATA = Path("/content")

# Load policy files from /content
POL = {
    p.stem.replace("sm-pol-", "").upper(): p.read_text(encoding="utf-8")
    for p in sorted(DATA.glob("sm-pol-*.txt"))
}

print("Policies loaded:", len(POL))
print("Policy IDs:", list(POL.keys()))

# Build the system prompt using the actual policy text
SYSTEM_PROMPT = [
    ("system", f"Policy ID: {policy_id}\n{policy_text}")
    for policy_id, policy_text in POL.items()
]
SYSTEM_PROMPT.append((
    "system",
    "Answer using the supplied workplace policies. Cite the relevant policy ID. "
    "If the policies do not contain the answer, say NOT FOUND."
))

# Staff question
Q = "How many unused leave days can I carry forward?"

# Mock model: no API key required
llm = get_llm(
    responses=["Up to 5 unused days carry forward (SM-POL-01)."]
)

# Call the model
reply = llm.invoke(SYSTEM_PROMPT + [("human", Q)])

# Extract the answer text
my_answer = reply.content

answer = pick(
    "d1-01-call",
    my_answer,
    is_using_solutions=USE_SOLUTIONS
)

check(
    "d1-01-call",
    isinstance(answer, str) and len(answer) > 0,
    "The text is in reply.content (a string)."
)

print("answer:")
display(Markdown("> " + answer.replace("\n", "\n> ")))

Policies loaded: 16
Policy IDs: ['01-ANNUAL-LEAVE', '02-IT-ACCESS', '03-EXPENSE-CLAIMS', '04-SAFETY-DRILL', '05-REMOTE-WORK - COPY', '05-REMOTE-WORK', '06-DATA-HANDLING - COPY', '06-DATA-HANDLING', '07-ONBOARDING - COPY', '07-ONBOARDING', '08-TRAVEL - COPY', '08-TRAVEL', '09-OVERTIME - COPY', '09-OVERTIME', '10-VENDOR-ONBOARDING - COPY', '10-VENDOR-ONBOARDING']
MOCK: canned replies
✔ d1-01-call: looks right
answer:


> Up to 5 unused days carry forward (SM-POL-01).

In [14]:
print("get_llm defined:", "get_llm" in globals())
print("USE_SOLUTIONS defined:", "USE_SOLUTIONS" in globals())

get_llm defined: True
USE_SOLUTIONS defined: True


In [15]:
from pathlib import Path

print("Current directory:", Path.cwd())
print("Files in current directory:")
print([p.name for p in Path.cwd().iterdir()])

print("\nFind policy files:")
print(list(Path("/content").rglob("sm-pol-*.txt"))[:20])

Current directory: /content
Files in current directory:
['.config', 'sm-pol-05-remote-work - Copy.txt', 'sm-pol-10-vendor-onboarding.txt', 'sm-pol-09-overtime - Copy.txt', 'sm-pol-07-onboarding.txt', 'sm-pol-07-onboarding - Copy.txt', 'sm-pol-02-it-access.txt', 'sm-pol-04-safety-drill.txt', 'sm-pol-06-data-handling - Copy.txt', 'sm-pol-05-remote-work.txt', '_groq_model_per_million_prices.json', 'sm-pol-09-overtime.txt', 'sm-pol-01-annual-leave.txt', 'sm-pol-06-data-handling.txt', 'sm-pol-08-travel - Copy.txt', 'sm-pol-03-expense-claims.txt', 'sm-pol-08-travel.txt', 'sm-pol-10-vendor-onboarding - Copy.txt', 'sample_data']

Find policy files:
[PosixPath('/content/sm-pol-05-remote-work - Copy.txt'), PosixPath('/content/sm-pol-10-vendor-onboarding.txt'), PosixPath('/content/sm-pol-09-overtime - Copy.txt'), PosixPath('/content/sm-pol-07-onboarding.txt'), PosixPath('/content/sm-pol-07-onboarding - Copy.txt'), PosixPath('/content/sm-pol-02-it-access.txt'), PosixPath('/content/sm-pol-04-safety

In [16]:
from pathlib import Path

DATA = Path("/content")

POL = {
    p.name[9:].upper(): p.read_text(encoding="utf-8")
    for p in sorted(DATA.glob("sm-pol-*.txt"))
}

print("Policies loaded:", len(POL))
print(list(POL.keys()))

Policies loaded: 16
['-ANNUAL-LEAVE.TXT', '-IT-ACCESS.TXT', '-EXPENSE-CLAIMS.TXT', '-SAFETY-DRILL.TXT', '-REMOTE-WORK - COPY.TXT', '-REMOTE-WORK.TXT', '-DATA-HANDLING - COPY.TXT', '-DATA-HANDLING.TXT', '-ONBOARDING - COPY.TXT', '-ONBOARDING.TXT', '-TRAVEL - COPY.TXT', '-TRAVEL.TXT', '-OVERTIME - COPY.TXT', '-OVERTIME.TXT', '-VENDOR-ONBOARDING - COPY.TXT', '-VENDOR-ONBOARDING.TXT']


In [17]:
from pathlib import Path

DATA = Path("/content/corpus")

POL = {
    p.name[9:].upper(): p.read_text(encoding="utf-8")
    for p in sorted(DATA.glob("sm-pol-*.txt"))
}

print("Policies loaded:", len(POL))
print(list(POL.keys()))

Policies loaded: 0
[]


**Step 2 - the automatic check. (●)** Write a function that returns True when the reply names `SM-POL-01` and the number 5, in any letter case.

In [18]:
  # Your turn: build the automatic check that tells the loop when an answer is good enough.
  def my_is_done(text):
      return None  # TODO: True if 'sm-pol-01' and '5' are both in text.lower()


  # Hand your function over here when you are ready to test it.
  my_attempt = None  # when ready, write: my_attempt = my_is_done

  # Use your check, or the reference check if you left the blank empty.
  is_done = pick("d1-01-check", my_attempt, is_using_solutions=USE_SOLUTIONS)

  # Try the check on one good reply and one bad reply.
  good, bad = is_done("Up to 5 days (SM-POL-01)."), is_done("Please ask HR.")

  # Self-check: a good reply passes, and replies missing the id or the number 5 fail.
  check(
      "d1-01-check",
      good and not bad and not is_done("SM-POL-01 only"),
      "Return True only when BOTH the id and the 5 are present.",
  )

✔ d1-01-check: looks right


True

In [19]:
# Your turn: build the automatic check that tells the loop when an answer is good enough.
def my_is_done(text):
    t = text.lower()
    return "sm-pol-01" in t and "5" in t


# Hand your function over here when you are ready to test it.
my_attempt = my_is_done

# Use your check, or the reference check if you left the blank empty.
is_done = pick("d1-01-check", my_attempt, is_using_solutions=USE_SOLUTIONS)

# Try the check on one good reply and one bad reply.
good, bad = is_done("Up to 5 days (SM-POL-01)."), is_done("Please ask HR.")

# Self-check: a good reply passes, and replies missing the id or the number 5 fail.
check(
    "d1-01-check",
    good and not bad and not is_done("SM-POL-01 only"),
    "Return True only when BOTH the id and the 5 are present.",
)

✔ d1-01-check: looks right


True

**Practice 1 - write the contract in words (●; no coding needed).** Fill in the three parts for this question: *How many unused leave days can I carry forward?* Keep the budget between 2 and 5 calls.

In [20]:
# Your turn: write the loop's contract in words before any code runs.
my_contract = 'one sentence, at least 5 words'  # replace None with a dict like the three lines below
# {'done': 'one sentence, at least 5 words',
#  'check': 'say in words how code will check it',
#  'max_calls': 3}

# Use your contract, or the reference one if you left the blank empty.
contract = pick("d1-01-contract", my_contract, is_using_solutions=USE_SOLUTIONS)
# Fall back to an empty dict so a wrong type fails the check instead of crashing.
c = contract if isinstance(contract, dict) else {}

# Test the contract: a real definition of done, a check described in words, and a sensible budget.
ok = (
    len(str(c.get("done", "")).split()) >= 5
    and len(str(c.get("check", ""))) > 5
    and 2 <= c.get("max_calls", 0) <= 5
)

# Self-check, then print the contract so you can reread it.
check("d1-01-contract", ok, "done: 5+ words; check: a sentence; max_calls: 2 to 5.")
print(contract)

✘ d1-01-contract: not yet. done: 5+ words; check: a sentence; max_calls: 2 to 5.
one sentence, at least 5 words


**Practice 2 - finish the loop (●).** Complete the loop function. It must stop as soon as the check is true, or when the budget is used up. The test below uses a model that is wrong first and right second, and a model that is always wrong.

In [21]:
# Your turn: finish the loop so it asks, checks, and stops on success or when the budget is used up.
def my_run_loop(llm, question, is_done, max_calls=3):
    for call in range(1, max_calls + 1):
        # Ask the model once for every call the budget allows.
        text = llm.invoke(SYSTEM_PROMPT + [("human", question)]).content

        #TODO 1: if is_done(text), return {'done': True, 'calls': call}

    #TODO 2: budget used up -> {'done': False, 'calls': max_calls}


# Hand your loop over here when you are ready to test it.
my_attempt = None  # when ready, write: my_attempt = my_run_loop

# Use your loop, or the reference loop if you left the blank empty.
run_loop = pick("d1-01-loop", my_attempt, is_using_solutions=USE_SOLUTIONS)
# A model that is wrong first and right second, to test stopping early.
slow = get_llm(responses=["Please ask HR.", "Up to 5 days (SM-POL-01)."])

# Run the loop on the two test models: one that succeeds on call 2, one that never does.
r1 = run_loop(slow, Q, is_done, 3)
r2 = run_loop(FakeListChatModel(responses=["Please ask HR."]), Q, is_done, 3)

# Show both results so you can compare them with the expected outcomes.
print(r1, r2, sep="\n")

MOCK: canned replies
{'done': True, 'calls': 2, 'replies': ['Please ask HR.', 'Up to 5 days (SM-POL-01).']}
{'done': False, 'calls': 3, 'replies': ['Please ask HR.', 'Please ask HR.', 'Please ask HR.']}


In [22]:
# Your turn: finish the loop so it asks, checks, and stops on success or when the budget is used up.
def my_run_loop(llm, question, is_done, max_calls=3):
    replies = []
    for call in range(1, max_calls + 1):
        # Ask the model once for every call the budget allows.
        text = llm.invoke(SYSTEM_PROMPT + [("human", question)]).content
        replies.append(text)

        # TODO 1: stop as soon as the check passes
        if is_done(text):
            return {"done": True, "calls": call, "replies": replies}

    # TODO 2: budget used up
    return {"done": False, "calls": max_calls, "replies": replies}


# Hand your loop over here when you are ready to test it.
my_attempt = my_run_loop

# Use your loop, or the reference loop if you left the blank empty.
run_loop = pick("d1-01-loop", my_attempt, is_using_solutions=USE_SOLUTIONS)
# A model that is wrong first and right second, to test stopping early.
slow = get_llm(responses=["Please ask HR.", "Up to 5 days (SM-POL-01)."])

# Run the loop on the two test models: one that succeeds on call 2, one that never does.
r1 = run_loop(slow, Q, is_done, 3)
r2 = run_loop(FakeListChatModel(responses=["Please ask HR."]), Q, is_done, 3)

# Show both results so you can compare them with the expected outcomes.
print(r1, r2, sep="\n")

MOCK: canned replies
{'done': True, 'calls': 2, 'replies': ['Please ask HR.', 'Up to 5 days (SM-POL-01).']}
{'done': False, 'calls': 3, 'replies': ['Please ask HR.', 'Please ask HR.', 'Please ask HR.']}


In [23]:
# The first model should succeed within the budget.
ok1 = isinstance(r1, dict) and r1.get("done") is True and r1.get("calls", 9) <= 3

# In mock mode the replies are canned, so the exact number of calls is known.
if LLM_MODE == "mock":
    ok1 = ok1 and r1.get("calls") == 2  # canned: wrong on call 1, right on call 2

# The always-wrong model must stop exactly at the budget and report not done.
ok2 = isinstance(r2, dict) and r2.get("done") is False and r2.get("calls") == 3

# Self-check both behaviours: stop on success, and stop at the budget.
check("d1-01-loop", ok1, "Return done=True as soon as is_done(text) is true.")
check("d1-01-loop-budget", ok2, "After max_calls wrong replies, return done=False.")

✔ d1-01-loop: looks right
✔ d1-01-loop-budget: looks right


True

### Check yourself

Answer in your head or on paper, then open the "answer" to validate your responses.

1. Name the three things a loop needs before it runs. <details><summary>Answer:</br></br></summary>A **definition of done**, an automatic **check**, and a **stop** budget.</br></br></details>

2. Your <u>check never says `True`</u>. What happens, and why is that safe? <details><summary>Answer:</br></br></summary>The loop stops at the budget and reports done = `False`. The budget is the safety net.</br></br></details>

3. Where is the answer text in what `llm.invoke(...)` gives back? <details><summary>Answer:</br></br></summary>In `reply.content`.</br></br></details>

4. Why is `mock` mode useful, and what can it not tell you? <details><summary>Answer</br></br></summary>It tests your code <u>offline</u> for free. It cannot tell you how good a real model is.<br></br></details>

In [24]:
# Names that Part B depends on from Part A.
need = ["DATA", "POL", "Q", "is_done", "run_loop"]

# Find which of those names are missing, for example if Part A was skipped.
missing = [n for n in need if n not in globals()]
# Report readiness so you know whether to go on.
print("All set for Part B." if not missing else "Missing: " + ", ".join(missing))

# Point late learners to the cells they need to run.
if missing and "d1-01-A" not in CODES:
    print("Click the heading Part A, Run > Run All Above Selected Cell, then rerun.")

# If Part A was run, rebuild the missing names from the reference code.
elif missing:
    fast_forward("d1-01-A")  # rebuilds the names above from reference code

# Quick sanity checks that the Part A data and check function are usable.
print("Policies loaded:", len(POL))
print("is_done works:", bool(is_done("5 days SM-POL-01")))

All set for Part B.
Policies loaded: 0
is_done works: True


## Part B - Tokens and Cost in Ringgit

![A yellow taxi stands in front of two tall towers in an imaginary city. A sentence is cut into numbered token tiles, and a taxi-style meter turns the tokens in and out into a price in RM. The price is a placeholder.](https://github.com/saifulak/EAE/blob/main/LLM/media/d1-01-token-meter.svg?raw=1)

### Idea: Tokens are the Taxi Meter (●)

- A **token** is a small piece of text, often a short word or part of a word. Models read and write tokens, not letters. In English, about 4 characters make 1 token. (**This is only an estimate*)
- A live provider sends a **usage record** with exact counts. In LangChain it is `message.usage_metadata` with `input_tokens` and `output_tokens`. Mock models send none, so we estimate.
- **Cost** = input tokens times input price + output tokens times output price. Prices are quoted per 1 million tokens. The meter in the picture shows one example call: 120 tokens in and 80 out, at the placeholder prices below (0.50 and 2.00 RM per 1 million), cost about RM 0.00022.

> **PLACEHOLDER PRICES.** Every price in this notebook is invented for practice. Look up your provider's current price list before you plan a real budget.

**Step 1 - estimate tokens (●).** Write the estimator.

In [25]:
# Your turn: write a rough token counter, since mock models report no usage.
def my_estimate(text):
    return None  # TODO: about 1 token per 4 characters; never less than 1


# Hand your function over here when you are ready to test it.
my_attempt = None  # when ready, write: my_attempt = my_estimate
# Use your estimator, or the reference one if you left the blank empty.
estimate_tokens = pick("d1-01-estimate", my_attempt, is_using_solutions=USE_SOLUTIONS)

# Self-check: a long text gives a predictable count, and empty text still counts as 1.
check(
    "d1-01-estimate",
    estimate_tokens("a" * 400) == 100 and estimate_tokens("") == 1,
    "Use len(text) // 4 and keep the answer at least 1.",
)

# Try it on a short sentence to see the scale.
print("Hello Malaysia ->", estimate_tokens("Hello Malaysia"), "tokens")

✔ d1-01-estimate: looks right
Hello Malaysia -> 3 tokens


**Step 2 - read the usage record, or estimate (●).** The function gives back `(tokens_in, tokens_out, source)`. The first message below is hand-made to show what a live record looks like.

In [26]:
# Your turn: report token counts from the usage record when there is one, and estimate otherwise.
def my_usage_of(msg, prompt):
    um = msg.usage_metadata
    if um:
        return um["input_tokens"], um["output_tokens"], "usage_metadata"
    return None  # TODO: no record, so estimate both sides: (in, out, 'estimate')


# Hand your function over here when you are ready to test it.
my_attempt = None  # when ready, write: my_attempt = my_usage_of
# Use your function, or the reference one if you left the blank empty.
usage_of = pick("d1-01-usage", my_attempt, is_using_solutions=USE_SOLUTIONS)

# A hand-made message that carries a usage record, like a live provider would send.
made = AIMessage(
    content="Five days.",
    usage_metadata={"input_tokens": 1000, "output_tokens": 500, "total_tokens": 1500},
)

# Compare a message with a record against one without.
a, b = usage_of(made, "x"), usage_of(AIMessage(content="Five days."), "How many days?")

# Self-check: the record is used when present, and the source is labelled 'estimate' when it is not.
check(
    "d1-01-usage",
    a == (1000, 500, "usage_metadata") and b and b[2] == "estimate",
    'Without a record: estimate both sides, source "estimate".',
)

# Show both results to see how the source label differs.
print(a, b)

✔ d1-01-usage: looks right
(1000, 500, 'usage_metadata') (3, 2, 'estimate')


**Practice - price a small batch in RM (●).** First the **PLACEHOLDER** price table (invented, not real). Then write `cost_myr` and run the batch check.

In [36]:
# PLACEHOLDER prices in RM per 1 million tokens. NOT REAL. Use your own list.
PRICES_MYR_PER_1M = {"input": model_input_price, "output": model_output_price}
# Print the prices so everyone sees they are placeholders, not real rates.
print("Using PLACEHOLDER prices (not real):", PRICES_MYR_PER_1M)


# Your turn: write the cost formula for one call, using the price table.
def my_cost(tokens_in, tokens_out, prices=PRICES_MYR_PER_1M):
    # return None  # TODO: tokens_in x input price + tokens_out x output price, per 1M
    return (tokens_in * prices["input"] + tokens_out * prices["output"]) / 1_000_000

# Hand your function over here when you are ready to test it.
my_attempt = my_cost  # when ready, write: my_attempt = my_cost
# Use your function, or the reference one if you left the blank empty.
cost_myr = pick("d1-01-cost", my_attempt, is_using_solutions=USE_SOLUTIONS)
# Price a known example call.
c1 = cost_myr(1000, 500)

# Self-check: the example matches the expected cost and zero tokens cost zero.
check(
    "d1-01-cost",
    c1 is not None and abs(c1 - 0.0015) < 1e-9 and cost_myr(0, 0) == 0,
    "Multiply each side by its price, add them, divide by 1,000,000.",
)

Using PLACEHOLDER prices (not real): {'input': 0.5, 'output': 2.0}
✔ d1-01-cost: looks right


True

In [35]:
# Three fictional helpdesk questions to price as a small batch.
QS = [
    "How many leave days carry forward?",
    "What is the meal allowance per day?",
    "How early must I book a flight?",
]

# Canned replies, one per question, so the batch is repeatable offline.
canned = [
    "Up to 5 days (SM-POL-01).",
    "RM 60 a day (SM-POL-03).",
    "14 days ahead (SM-POL-08).",
]

# Start a fresh mock model and a running total for the batch.
llm, total = get_llm(responses=canned), 0.0
# For each question, get the token counts and the cost, then add it to the total.
for q in QS:
    t_in, t_out, src = usage_of(llm.invoke(SYSTEM_PROMPT + [("human", q)]), q) or (
        0,
        0,
        "no result",
    )
    c = cost_myr(t_in, t_out) or 0.0
    total += c
    # Print one row per question so the numbers can be compared.
    print(f"{q[:34]:<36}in={t_in:<3} out={t_out:<3} {src:<9} RM {c:.7f}")

# Show the batch total and a rough monthly projection, using placeholder prices.
print(f"Batch of 3: RM {total:.7f} (PLACEHOLDER prices, estimated tokens)")
print(f"2,000 such questions a month: about RM {total / 3 * 2000:.4f}")

# Self-check: every row should have a cost above zero.
check("d1-01-batch", total > 0, "Each row needs a cost above 0: fix a function.")

MOCK: canned replies
How many leave days carry forward?  in=8   out=6   estimate  RM 0.0000000
What is the meal allowance per day  in=8   out=6   estimate  RM 0.0000000
How early must I book a flight?     in=7   out=6   estimate  RM 0.0000000
Batch of 3: RM 0.0000000 (PLACEHOLDER prices, estimated tokens)
2,000 such questions a month: about RM 0.0000
✘ d1-01-batch: not yet. Each row needs a cost above 0: fix a function.


False

### Check yourself

1. About how many tokens is a 400-character English text? <details><summary>Answer:</br></br></summary>About 100 ([one token is roughly 4 characters](https://platform.openai.com/tokenizer)). It is an estimate.</br></br></details>
2. Why do we say the prices are PLACEHOLDER? <details><summary>Answer:</br></br></summary>They are invented for practice. Real prices change, so read your provider's list.</br></br></details>
3. Why does the batch table say `estimate`? <details><summary>Answer:</br></br></summary>Mock models send no usage record, so we estimate from text length.</br></br></details>
4. Is the RM figure a bill? <details><summary>Answer:</br></br></summary>No. It is a planning estimate using invented prices.</br></br></details>

You should now have `is_done`, `run_loop`, `estimate_tokens`, `usage_of`, `cost_myr` and the placeholder prices.

In [29]:
# Everything Part B should have produced.
need = [
    "is_done",
    "run_loop",
    "estimate_tokens",
    "usage_of",
    "cost_myr",
    "PRICES_MYR_PER_1M",
]

# List anything missing and say whether Part B is complete.
missing = [n for n in need if n not in globals()]
print("d1-01 complete." if not missing else "Missing: " + ", ".join(missing))

# Point late learners to the cells they need to run.
if missing and not all(p in CODES for p in ("d1-01-A", "d1-01-B")):
    print("Click the heading Part A, Run > Run All Above Selected Cell, then rerun.")

# If the Parts were run, rebuild the missing names from the reference code.
elif missing:
    fast_forward("d1-01-B")  # rebuild variable names from reference code

# Final sanity check: price a known call with your cost function.
print("Example cost of 1,000 tokens in and 500 out: RM", cost_myr(1000, 500))

d1-01 complete.
Example cost of 1,000 tokens in and 500 out: RM 0.0015


## Recap and next step
- A finishable loop has a **definition of done**, an **automatic check** and a **stop budget**.
- `get_llm()` gives you a model. Mock replies are canned. Live replies are real but need a human check.
- Tokens are the meter:
  
  $$\text{cost} = n(\text{token}_{\text{in}}) \cdot \text{price}_{\text{in}} + n(\text{token}_{\text{out}}) \cdot \text{price}_{\text{out}}$$

  Prices here (*in this notebook*) are placeholders.

- Next: **[d1-02-prompts-that-behave.ipynb](d1-02-prompts-that-behave.ipynb)** (templates, roles and examples so the helper answers in a fixed shape).

### switch to live Groq (◆)

1. Put your key in the `.env` file described in PRE-WORK. Set `LLM_MODE=groq` and `GROQ_MODEL=openai/gpt-oss-20b` (suggested, or [else](https://console.groq.com/docs/models)). Restart the kernel and run the cells again.
2. The course is designed with free-plan in mind. If you see a rate-limit error, wait and retry with a longer pause each time.
3. Remember the three rules: no personal or company data, and a human checks the answer.

### streaming (◆)

`llm.invoke` waits for the whole answer. `llm.stream` gives **chunks** as they arrive. It does not make the model faster. It makes the wait feel shorter. The mock streams one character at a time. The cell after the retry helper shows it.

In [30]:
# Retry helper: if the call fails (for example a rate limit), wait longer each time before trying again.
def ask_with_retry(llm, text, tries=3):
    for n in range(tries):
        try:
            return llm.invoke(SYSTEM_PROMPT + [("human", text)]).content
        except Exception as err:  # for example a rate-limit error
            wait = 2 ** (n + 1)
            print("Waiting", wait, "s after", type(err).__name__)
            time.sleep(wait)
    return None


# Try the helper with a mock model and show the reply as a quoted block.
display(
    Markdown(
        "> "
        + (
            _first_response := ask_with_retry(
                get_llm(responses=["Hello from the helpdesk."]), "Hi"
            )
        )
    )
)

# Show the new question that will be streamed.
print("ask:", end="")
question: str = "Where can I apply for leave?"
display(Markdown("> " + question))

# Remember when the first chunk arrives, so the delay between chunks can be shown.
_timer: datetime

# Stream the answer chunk by chunk, so the wait feels shorter, and collect the pieces.
pieces, t0 = [], time.perf_counter()
for _i, chunk in enumerate(
    get_llm(responses=["Leave is applied in the HR portal."]).stream(
        SYSTEM_PROMPT + [("ai", _first_response), ("human", question)]
    )
):
    if not _i:
        _timer = datetime.now()
        print("\nanswer:", end=f"\t{_timer}\n\n")

    pieces.append(_chunked_response := chunk.content)

    if _chunked_response.startswith(" "):
        print(
            f"(+{round(1e3 * (datetime.now() - _timer).total_seconds(), 3)}ms)", end=""
        )

    print(_chunked_response, end="")

# Show the full answer once streaming is finished, and how many chunks it took.
display(Markdown("> " + "".join(pieces).strip().replace("\n", "\n> ")))
print("[", len(pieces), "chunks]")

MOCK: canned replies


> Hello from the helpdesk.

ask:

> Where can I apply for leave?

MOCK: canned replies

answer:	2026-10-26 06:04:46.662630

Leave(+0.253ms) is(+0.39ms) applied(+1.273ms) in(+2.221ms) the(+2.554ms) HR(+2.707ms) portal.

> Leave is applied in the HR portal.

[ 34 chunks]


### Notes for Later (■)

- **Local model (an optional idea).** You may run a small model on your own machine with HuggingFace. It needs heavy packages (torch and transformers) that may not fit 4 GiB RAM and 2 GiB disk limits. If you are curious, you may explore this alternative.
- **Price-table variants.** Change the numbers in `PRICES_MYR_PER_1M` and see how the batch cost moves. Try a table where output tokens cost 8 times input tokens.
- **Honesty labels.** Mock replies are canned. Prices are PLACEHOLDER. Groq limits are supplied [here, officially](https://console.groq.com/docs/rate-limits).

- **Hugging Face practice (separate environment):** Continue with `d1-05-huggingface-pretrained-models.ipynb` for pretrained inference, then `d1-06-fine-tuning-sentiment-analysis.ipynb` for actual weight updates. These are separate from this API-focused course environment because `.requirements.txt` intentionally excludes `torch` and `transformers`.
- **Web app deployment:** Use `d1-07-deploy-llm-app.ipynb` with the supplied `app.py` and `requirements-streamlit.txt`.
